# 4. Modelos

← [Pré-processamento e data augmentation](03_preprocessing.ipynb) · [índice](README.pt-BR.md) · [Treino e experimentos](05_training.ipynb) →

As três famílias de arquiteturas comparadas: MLP, CNN no estilo LeNet e CNN no estilo VGG.

## Modelos

Algumas escolhas valem para todas as arquiteturas:
- **ReLU** como ativação;
- **inicialização He** (Kaiming), que ajusta a escala dos pesos ao tamanho da entrada e é a indicada para ReLU;
- a saída tem 10 *scores* sem normalização, e a **softmax com cross-entropy** fica na função de perda.

**MLP.** A imagem é achatada em um vetor de 3072 valores e passa por camadas totalmente conectadas.
Com isso, o modelo ignora a estrutura espacial da imagem.

**CNN baseline (estilo LeNet).** Dois blocos de convolução 5×5, ReLU e pooling, seguidos de duas camadas FC:
`[CONV 5×5 → ReLU → POOL]×2 → FC → FC`.

**CNN estilo VGG.** Usa só convoluções 3×3 com stride 1 e pad 1, que preservam o tamanho
(`(N + 2P − F)/S + 1 = N`), e max pooling 2×2 com stride 2, que reduz o tamanho pela metade, como na
VGGNet. Com 3 blocos, a resolução cai de 32×32 para 16×16, depois 8×8 e por fim 4×4. Segue a tendência de usar
filtros menores e arquiteturas mais profundas. Duas convoluções 3×3 seguidas enxergam
a mesma região que uma 5×5, mas com menos parâmetros (2·9·C² contra 25·C²) e uma ReLU a mais, que é o
argumento do artigo da VGG.

**Contagem de parâmetros.** Cada filtro tem F·F·C_in pesos mais 1 bias, e o pooling
não tem parâmetros. Na E4, as convoluções somam 287.008 parâmetros e as camadas FC somam
4·4·128·256 + 256 + 256·10 + 10 = 527.114. O total de 814.122 bate exatamente com o que o PyTorch mostra abaixo.
Assim como na VGGNet, a maior parte dos parâmetros (65%) está nas camadas FC.

**BatchNorm.** Fica depois da conv ou da FC e antes da ReLU. Quando há BatchNorm,
a camada anterior fica sem bias, porque o deslocamento β aprendido pelo BN já faz esse papel. No teste
(`model.eval()`), o BN usa as médias e variâncias acumuladas durante o treino (médias móveis), e não as do
batch.

**Dropout.** Fica nas camadas FC, com p = 0,5 como na AlexNet. Também uso um
dropout leve (p = 0,1) por canal nos blocos convolucionais. No PyTorch, p é a
probabilidade de zerar um neurônio. Na formulação original do dropout, a probabilidade é a de manter o neurônio, ou
seja, 1 − p nessa notação, e as ativações são multiplicadas por 1 − p no teste. O PyTorch usa o *inverted
dropout*, que divide por 1 − p no treino e não mexe em nada no teste. As duas versões são equivalentes, porque a saída média no treino fica
igual à saída no teste.

In [1]:
import inspect
from IPython.display import Code, display

def show(*objs):
    """Mostra o código-fonte das funções/classes (definidas em svhn_utils.py)."""
    display(Code("\n\n".join(inspect.getsource(o) for o in objs), language="python"))

import pandas as pd
import torch

from svhn_utils import EXPERIMENTS, MLP, VGGNet, SimpleCNN, build_model, count_params, he_init

show(he_init, MLP, SimpleCNN, VGGNet)

def he_init(model):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return model


class MLP(nn.Module):
    # 3072 -> camadas ocultas -> 10, opcionalmente com BatchNorm e Dropout
    def __init__(self, hidden=(512, 256), batchnorm=False, dropout=0.0):
        super().__init__()
        layers, d = [nn.Flatten()], 32 * 32 * 3
        for h in hidden:
            layers.append(nn.Linear(d, h, bias=not batchnorm))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, 10))
        self.net = nn.Sequential(*layers)
        he_init(self)

    def forward(self, x):
        return self.net(x)


class SimpleCNN(nn.Module):
    # 32x32x3 -conv5x5,pad2-> 32x32x32 -pool-> 16x16x32 -conv5x5,pad2-> 16x16x64 -pool-> 8x8x64 -> FC256 -> FC10
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(8 * 8 * 64, 256), nn.ReLU(inplace=True), nn.Linear(256, 10))
        he_init(self)

    def forward(self, x):
        return self.classifier(self.features(x))


class VGGNet(nn.Module):
    # cada bloco = convs_per_block x [CONV3x3 -> (BN) -> ReLU] + MaxPool2x2 (+ Dropout2d opcional)
    # classificador = (Dropout) -> FC oculta -> (BN) -> ReLU -> (Dropout) -> FC 10
    def __init__(self, channels=(32, 64, 128), convs_per_block=2, batchnorm=False,
                 dropout=0.0, conv_dropout=0.0, fc=256):
        super().__init__()
        layers, c_in = [], 3
        for c in channels:
            for _ in range(convs_per_block):
                layers.append(nn.Conv2d(c_in, c, 3, padding=1, bias=not batchnorm))
                if batchnorm:
                    layers.append(nn.BatchNorm2d(c))
                layers.append(nn.ReLU(inplace=True))
                c_in = c
            layers.append(nn.MaxPool2d(2))
            if conv_dropout > 0:
                layers.append(nn.Dropout2d(conv_dropout))
        self.features = nn.Sequential(*layers)
        spatial = 32 // (2 ** len(channels))
        head = [nn.Flatten()]
        if dropout > 0:
            head.append(nn.Dropout(dropout))
        head.append(nn.Linear(c_in * spatial * spatial, fc, bias=not batchnorm))
        if batchnorm:
            head.append(nn.BatchNorm1d(fc))
        head.append(nn.ReLU(inplace=True))
        if dropout > 0:
            head.append(nn.Dropout(dropout))
        head.append(nn.Linear(fc, 10))
        self.classifier = nn.Sequential(*head)
        he_init(self)

    def forward(self, x):
        return self.classifier(self.features(x))

In [2]:
# verificação das dimensões
for name, m in [("MLP", MLP()), ("SimpleCNN", SimpleCNN()), ("VGGNet", VGGNet(batchnorm=True, dropout=0.5))]:
    out = m.eval()(torch.zeros(2, 3, 32, 32))
    print(f"{name:10s} saída {tuple(out.shape)}  parâmetros: {count_params(m):,}")

MLP        saída (2, 10)  parâmetros: 1,707,274
SimpleCNN  saída (2, 10)  parâmetros: 1,105,098
VGGNet     saída (2, 10)  parâmetros: 814,826


## Arquitetura de cada experimento

In [3]:
pd.DataFrame([{"Exp.": e["id"], "Modificação": e["desc"], "Arquitetura": e["arch"],
               "Configuração": e["model_kw"], "Parâmetros": f"{count_params(build_model(e['arch'], **e['model_kw'])):,}"}
              for e in EXPERIMENTS])

,Exp.,Modificação,Arquitetura,Configuração,Parâmetros
0,E1,MLP baseline (3072-512-256-10),mlp,"{'hidden': (512, 256)}","1,707,274"
1,E2,MLP + BatchNorm + Dropout 0.3,mlp,"{'hidden': (512, 256), 'batchnorm': True, 'dro...","1,708,042"
2,E3,CNN baseline (LeNet-like),simple_cnn,{},"1,105,098"
3,E4,CNN VGG-like (3 blocos × 2 conv3x3),vgg,"{'channels': (32, 64, 128), 'convs_per_block': 2}","814,122"
4,E5,+ BatchNorm,vgg,"{'channels': (32, 64, 128), 'convs_per_block':...","814,826"
5,E6,"+ Dropout (0.5 FC, 0.1 conv)",vgg,"{'channels': (32, 64, 128), 'convs_per_block':...","814,826"
6,E7,+ Data Augmentation,vgg,"{'channels': (32, 64, 128), 'convs_per_block':...","814,826"
7,E8,SGD+Momentum 0.9 (lr 0.05),vgg,"{'channels': (32, 64, 128), 'convs_per_block':...","814,826"
8,E9,SGD sem momentum (lr 0.05),vgg,"{'channels': (32, 64, 128), 'convs_per_block':...","814,826"
9,E10,+ LR decay (÷10 no platô),vgg,"{'channels': (32, 64, 128), 'convs_per_block':...","814,826"


## Modelo final (E16)

In [4]:
print(build_model("vgg", channels=(64, 128, 256), convs_per_block=2, batchnorm=True, dropout=0.5, conv_dropout=0.1))

VGGNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout2d(p=0.1, inplace=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (12): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  